# 03 - Inpainting and relighting

The finished recipe lives in `src/pipeline.py`; this notebook is the record of the experiments behind it.

**Inpainting** changes the generation equation: at every denoising step the region outside the mask is replaced with the correspondingly noised original, so product pixels are copied rather than generated. This is what finally fixes the identity problem (the invented gold rim from notebook 01).

Mask engineering: invert (white = regenerate), erode 7 px, feather 21 px. The canny map is cropped to the product region so no ghost of the old background survives, and the ControlNet scale drops to 0.6 because geometry is now protected twice.

**IC-Light** is integrated by hand: SD 1.5 UNet, `conv_in` widened 4 -> 8 channels, the released offset added to the weights, and a forward hook that injects the foreground latent. Light direction only becomes controllable with hybrid conditioning - the concat latent must be the product on grey, not the lit scene.

In [ ]:
import os, sys
sys.path.insert(0, "../src")
os.chdir("..")
print("working dir:", os.getcwd())

## 1. Inpainting pipeline

In [ ]:
import gc
import cv2
import numpy as np
import torch
from PIL import Image
from diffusers import (StableDiffusionControlNetInpaintPipeline, ControlNetModel,
                       UniPCMultistepScheduler)

controlnet = ControlNetModel.from_pretrained(
    "lllyasviel/sd-controlnet-canny", torch_dtype=torch.float16)
pipe = StableDiffusionControlNetInpaintPipeline.from_pretrained(
    "runwayml/stable-diffusion-inpainting", controlnet=controlnet,
    torch_dtype=torch.float16, safety_checker=None)
pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)
pipe = pipe.to("cuda")
print("inpaint pipeline ready")

## 2. Masks and cropped canny

The first run left a dark halo around the mug: rembg's mask is slightly wider than the product, so the overshoot was treated as *product* and carried old table pixels into the new scene.

In [ ]:
PROMPT = ("professional product photo of a white ceramic mug on a marble "
          "kitchen counter, soft studio lighting, high quality")
NEGATIVE = "blurry, low quality, distorted, deformed, watermark, text, logo, signature"

source = Image.open("inputs/testset/01_matte_mug.jpeg").convert("RGB").resize((512, 640))
product_mask = np.array(Image.open("outputs/mask_rembg.png").convert("L"))

# invert + feather
inpaint_mask = cv2.GaussianBlur(255 - product_mask, (21, 21), 0)

# crop canny to the product region (dilate first so the silhouette line survives)
canny_full = np.array(Image.open("outputs/canny_final.png").convert("L"))
widened = cv2.dilate(product_mask, np.ones((15, 15), np.uint8))
canny_cropped = np.where(widened > 127, canny_full, 0).astype(np.uint8)
control_image = Image.fromarray(np.stack([canny_cropped] * 3, axis=-1))
control_image.save("outputs/canny_cropped.png")

## 3. Erosion sweep (0 / 5 / 11 / 21 px)

Trade-off: too little erosion leaves the halo, too much pushes the real product edge into the generated region and identity leaks back in. Selected value: **7 px**.

In [ ]:
for erode_px in [0, 5, 7, 11, 21]:
    mask = (cv2.erode(product_mask, np.ones((erode_px, erode_px), np.uint8))
            if erode_px else product_mask)
    feathered = cv2.GaussianBlur(255 - mask, (21, 21), 0)

    generator = torch.Generator(device="cuda").manual_seed(42)
    result = pipe(prompt=PROMPT, negative_prompt=NEGATIVE, image=source,
                  mask_image=Image.fromarray(feathered), control_image=control_image,
                  num_inference_steps=25, controlnet_conditioning_scale=0.6,
                  generator=generator, height=640, width=512).images[0]
    result.save(f"outputs/inpaint_erode_{erode_px}.png")
    print(f"erode={erode_px} done")

## 4. Module check

Determinism turns refactor testing into pixel comparison: same recipe, same seed, so a correct refactor must reproduce the notebook result exactly.

In [ ]:
from pipeline import load_models, replace_background

del pipe, controlnet
gc.collect()
torch.cuda.empty_cache()

models = load_models()
result, mask = replace_background(
    "inputs/testset/01_matte_mug.jpeg", PROMPT, models, seed=42)
result.save("outputs/pipeline_check.png")

a = np.array(result).astype(int)
b = np.array(Image.open("outputs/inpaint_erode_7.png")).astype(int)
print(f"max pixel difference: {np.abs(a - b).max()}")

## 5. Generalisation to the hard cases

The transparent glass keeps its old interior but the neutral tones do not clash with the new scene; the headphones show the predicted failure - rembg filled the gap between the band and the cups, so a patch of the old table is frozen into the new one.

In [ ]:
for name, description in [("04_transparent_glass", "glass tumbler"),
                         ("05_lowcontrast_headphones", "black wireless headphones")]:
    result, _ = replace_background(
        f"inputs/testset/{name}.jpeg",
        f"professional product photo of {description} on a marble kitchen counter, "
        f"soft studio lighting, high quality",
        models, seed=42)
    result.save(f"outputs/pipeline_{name}.png")
    print(f"{name} done")

## 6. IC-Light integration

`load_iclight` performs the three steps described above. On 12 GB the inpainting pipeline has to be released first, otherwise the driver spills to system RAM and generation slows down roughly tenfold.

In [ ]:
from pipeline import load_iclight, relight

del models
gc.collect()
torch.cuda.empty_cache()

iclight = load_iclight()
print("IC-Light ready")

## 7. Naive vs hybrid conditioning

**Naive** (concat latent = the full lit scene): light prompts have almost no effect - the model copies the lighting it is shown. Raising `strength` to 0.85 and 0.95 changes nothing, which rules out strength as the cause.

**Hybrid** (concat latent = product on grey, img2img input = the scene): light control returns immediately. The cost is partial regeneration of the background texture.

In [ ]:
scene = Image.open("outputs/pipeline_check.png").convert("RGB")
mask_pil = Image.open("outputs/mask_rembg.png")

for direction in ["warm golden hour light from left side, soft shadows",
                  "soft light from right side, product photography"]:
    out = relight(scene, mask_pil, f"{PROMPT}, {direction}", iclight,
                  seed=42, strength=0.7)
    tag = direction.split()[-3].replace(",", "")
    out.save(f"outputs/relit_{tag}.png")
    print(f"{direction[:40]}... done")

## 8. End to end

`run_pipeline` chains all three stages with sequential loading. It returns both the relit result and the stage-2 scene, which is what the metrics notebook compares.

In [ ]:
del iclight
gc.collect()
torch.cuda.empty_cache()

from pipeline import run_pipeline

result, scene = run_pipeline(
    "inputs/testset/01_matte_mug.jpeg",
    scene_prompt=PROMPT,
    light_prompt=f"{PROMPT}, warm golden hour light from left side, soft shadows",
    seed=42)

scene.save("outputs/e2e_stage2.png")
result.save("outputs/e2e_final.png")
print("end-to-end run complete")